In [21]:
import pandas as pd
import numpy as np
import random
import math

# Recipe class


In [22]:
class Recipe:
    def __init__(self, RID, recipes_df):
        recipe_row = recipes_df[recipes_df['RID'] == RID].iloc[0]
        
        self.RID = RID
        self.name = recipe_row['Name']
        self.type = recipe_row['type']
        self.category = recipe_row['Category']
        self.total_price = recipe_row['total_price']
        self.provided_calories = recipe_row['provided_calories']

    def __eq__(self, other):
        return isinstance(other, Recipe) and self.RID == other.RID


### initialising recipes list

In [23]:
def initialize_recipes(recipes_df):
    return {rid: Recipe(rid, recipes_df) for rid in recipes_df['RID']}


recipes_df = pd.read_csv("./data/recipes.csv")
recipes = initialize_recipes(recipes_df)


# Node class

In [ ]:
class MealPlannerState:
    def __init__(self, day_number, meal_type, recipe_id, remaining_budget, today_calorie_use, used_meals):
        self.day = day_number
        self.meal_type = meal_type
        self.recipe_id = recipe_id
        self.remaining_budget = remaining_budget
        self.today_calorie_use = today_calorie_use
        self.used_meals = used_meals

    def __eq__(self, other):
        return isinstance(other, MealPlannerState) and \
            self.day == other.day and \
            self.meal_type == other.meal_type and \
            self.recipe_id == other.recipe_id

    def __hash__(self):
        return hash((self.day, self.meal_type, self.recipe_id))
        

In [25]:
class Node:    
    def __init__(self, state, parent=None, action=None, cost=0, heuristic=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = (parent.g + cost) if parent else 0
        self.f = self.g + heuristic
        self.depth = 0 if parent is None else parent.depth + 1

    def __lt__(self, other):
        return self.f < other.f

    def __eq__(self, other):
        return isinstance(other, Node) and self.state == other.state

    def __hash__(self):
        return hash(self.state)


# Problem class

In [ ]:
class MealPlanningProblem:

    def __init__(self, recipes, daily_calories_need, total_budget, num_days=7):
        self.recipes = recipes
        self.total_budget = total_budget
        self.daily_calories_need = daily_calories_need
        self.num_days = num_days
        self.meals_per_day = 3  # Breakfast, Lunch, Dinner
        self.total_slots = num_days * self.meals_per_day
        self.meal_types = ['Breakfast', 'Lunch', 'Dinner']

        self.meal_type_weights = {
            'Breakfast': 0.25,
            'Lunch': 0.40,
            'Dinner': 0.35
        }

        self.initial_state = MealPlannerState(0, 0, None, total_budget, 0, set())
        
        recipes_values = list(self.recipes.values())
        self._avg_price = sum(r.total_price for r in recipes_values) / len(recipes_values)

    def _get_remaining_days(self, state):
        current_slot = state.day * self.meals_per_day + state.meal_type + 1
        remaining_slots = self.total_slots - current_slot
        return math.ceil(remaining_slots / self.meals_per_day)

    def _filter_recipes(self, meal_type=None):
        filtered = []
        for rid, recipe in self.recipes.items():
            if meal_type and recipe.type.lower() != meal_type.lower():
                continue
            filtered.append(rid)
        return filtered

    def _get_all_recipes_from_path(self, node):
        recipes = []
        current = node
        while current.parent is not None:
            recipes.append(current.action)
            current = current.parent
        recipes.reverse()
        return recipes

    def _calculate_totals(self, recipes_list):
        total_cost = 0
        total_calories = 0
        
        for recipe_id in recipes_list:
            recipe = self.recipes[recipe_id]
            total_cost += recipe.total_price
            total_calories += recipe.provided_calories
        
        return total_cost, total_calories

    def is_goal(self, node):
        state = node.state
        
        if state.day < self.num_days:
            return False
        
        recipes_chosen = self._get_all_recipes_from_path(node)
        
        if len(recipes_chosen) != self.total_slots:
            return False
        
        total_cost, total_calories = self._calculate_totals(recipes_chosen)
        
        goal_calories = self.daily_calories_need * self.num_days
        calorie_tolerance = goal_calories * 0.1
        calories_ok = abs(total_calories - goal_calories) <= calorie_tolerance
        
        budget_ok = total_cost <= self.total_budget
        
        return calories_ok and budget_ok

    def get_valid_actions(self, state):
        meal_type = self.meal_types[state.meal_type]
        return self._filter_recipes(meal_type=meal_type)

    def expand_node(self, node, use_cost=True, use_heuristic=False):
        state = node.state
        
        if state.day >= self.num_days:
            return []
        
        children = []
        valid_actions = self.get_valid_actions(node.state)
        
        for recipe_id in valid_actions:

            if recipe_id in state.used_meals:
                continue

            recipe = self.recipes[recipe_id]
            
            action_cost = self.calculate_cost(node.state, recipe_id) if use_cost else 0
            
            new_meal_idx = (state.meal_type + 1) % self.meals_per_day
            new_day = state.day if new_meal_idx > 0 else state.day + 1
            new_remaining_budget = state.remaining_budget - recipe.total_price
            new_calorie_use = 0 if new_meal_idx == 0 else state.today_calorie_use + recipe.provided_calories
            
            if new_day % 4 == 0:
                new_used_meals = set({})
            else:
                new_used_meals = set(state.used_meals)

            new_used_meals.add(recipe_id)
            
            new_state = MealPlannerState(new_day, new_meal_idx, recipe_id, new_remaining_budget, new_calorie_use, new_used_meals)
            
            heuristic = self.calculate_heuristic(new_state) if use_heuristic else 0
            
            child = Node(new_state, parent=node, action=recipe_id, cost=action_cost, heuristic=heuristic)
            children.append(child)
        
        return children

    def calculate_cost(self, state, recipe_id=None):
        if recipe_id is None:
            recipe_id = state.recipe_id
        
        if recipe_id is None:
            return 0
        
        recipe = self.recipes[recipe_id]
        meal_type = self.meal_types[state.meal_type]
        
        remaining_days = self._get_remaining_days(state)
        
        allocated_price = self.meal_type_weights[meal_type] * (state.remaining_budget / max(1, remaining_days))
        allocated_calories = self.meal_type_weights[meal_type] * self.daily_calories_need 

        price_deviation = abs(recipe.total_price - allocated_price)
        calorie_deviation = abs(recipe.provided_calories - allocated_calories)
        
        normalized_price_dev = price_deviation / allocated_price if allocated_price > 0 else 0
        normalized_cal_dev = calorie_deviation / allocated_calories if allocated_calories > 0 else 0
        
        total_cost = (normalized_price_dev + normalized_cal_dev) / 2.0
        
        return total_cost


    def calculate_heuristic(self, state):
        remaining_slots = self.total_slots - (state.day * self.meals_per_day + state.meal_type)
        if remaining_slots <= 0:
            return 0

        ideal_spend = state.remaining_budget / remaining_slots

        price_gap = abs(self._avg_price - ideal_spend) / self._avg_price
        calorie_gap = abs(state.today_calorie_use - self.daily_calories_need) / self.daily_calories_need

        return (price_gap + calorie_gap) / 2.0


# Search class

In [27]:
import queue

class AstarSearch:
    def __init__(self,problem):
        self.problem = problem
        self.frontier = queue.PriorityQueue()

        self.explored = set()

    def search(self):

        node = Node(self.problem.initial_state)
        self.frontier.put(node)
        
        while True:
            if self.frontier.empty():
                return None

            node = self.frontier.get()
            
            if self.problem.is_goal(node):
                solution = self._get_solution_path(node)
                return solution

            self.explored.add(node.state)

            children = self.problem.expand_node(node, True, True)

            for child in children:
                if child.state not in self.explored and child not in self.frontier.queue:
                    self.frontier.put(child)

    def _get_solution_path(self, solution_node):
        path = []
        current = solution_node
        
        while current.parent is not None:
            path.append(current.action)
            current = current.parent
        
        path.reverse()
        
        path = [tuple([
                self.problem.recipes[path[i]].name,
                self.problem.recipes[path[i+1]].name,
                self.problem.recipes[path[i+2]].name
            ]) for i in range(0, len(path), 3)]

        return path


        

In [28]:
def test_a_star(TDEE, budget, days):
    problem = MealPlanningProblem(recipes, TDEE, budget, days)
    A_star = AstarSearch(problem)

    solution = A_star.search()

    if solution is None:
        print("couldn't find a suitable plan")
        return

    print(solution)
    

In [29]:
##### TESTING #####
test_a_star(2200, 6000, 7)

[('Protein peanaut butter Smoothie', 'Rice & Egg Bowl', 'Lham Lahlou'), ('Protein peanaut butter Smoothie', 'Tuna Rice Olive Bowl', 'Hchaichi'), ('Harissa Sardine Egg Plate', 'Chorba el Djaj', 'Harira'), ('Banana Peanut Protein Smoothie', 'Chorba Frik', 'Chicken Mchermel'), ('Protein peanaut butter Smoothie', 'Tuna Rice Olive Bowl', 'Lham Lahlou'), ('Harissa Sardine Egg Plate', 'Chorba Frik', 'Mutton Carrot Pea Dinner'), ('Protein peanaut butter Smoothie', 'Sardine Rice Tomato Bowl', 'Tlitli Chicken Dinner')]
